# Baseline Feature Instantiation

From baseline model.


In [0]:
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lead

fe = FeatureEngineeringClient()
CATALOG = "mlo"
noaa_table = f"{CATALOG}.weather_mlops.noaa_historical_daily"

# Bronze now holds ten stations and runs to the present (see data_pipelines/). v1 and v2
# features, the labels, the model card and train_baseline.py were all built when it held
# exactly one station and ended 2024-12-31.
#
# Both filters below exist to keep that true. Without the station filter, re-running this
# notebook would silently multiply both tables roughly tenfold; without the date filter it
# would extend them by ~20 months. Either would change what 03_baseline_model.ipynb and
# drift_monitoring.py read, with no error anywhere.
#
# v3 is where the extra stations and the extra date range get used.
PRIMARY_STATION = "GHCND:USW00014819"          # Chicago Midway — the prediction target
V1_START, V1_END = "2020-01-01", "2024-12-31"  # the original training window

bronze = (spark.table(noaa_table)
          .filter(col("station") == PRIMARY_STATION)
          .filter(col("date").between(V1_START, V1_END)))

# Feature table — AWND/TMAX/TMIN only (PRCP excluded → no leakage)
feat = bronze.select("station", "date", "AWND", "TMAX", "TMIN").na.drop(subset=["station", "date"])
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.features")
FT = f"{CATALOG}.features.weather_daily"
spark.sql(f"DROP TABLE IF EXISTS {FT}")
fe.create_table(name=FT, primary_keys=["date", "station"], timestamp_keys=["date"],
                df=feat, description="Daily NOAA features (AWND,TMAX,TMIN)")

# Labels (kept separate — this is NOT in the feature table)
w = Window.partitionBy("station").orderBy("date")     # per-station, chronological

labels = (bronze                                       # filtered — was re-reading the raw table
          .select("station", "date", "PRCP")
          .withColumn("Bad_today", (col("PRCP") > 0.5).cast("int"))
          .withColumn("Bad", lead("Bad_today", 1).over(w))   # ← Bad = TOMORROW's weather
          .na.drop(subset=["Bad"])                           # drop each station's last day
          .select("station", "date", "Bad"))

labels.write.mode("overwrite").saveAsTable(f"{CATALOG}.features.weather_labels")

print(f"Baseline features + labels built for {PRIMARY_STATION}, {V1_START}..{V1_END}")
print(f"rows: {feat.count()} features, {labels.count()} labels")

# Features v2
We add prior day precip, ensure we are able to pick up new features and log them in a new run (shows up in MLFlow differently.)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, avg

w = Window.partitionBy("station").orderBy("date")

# `bronze` is already filtered to PRIMARY_STATION and to V1_START..V1_END in the cell above,
# so v2 keeps exactly the shape drift_monitoring.py and 03_baseline_model.ipynb expect.
# Do not point this at the unfiltered table — v3 is where the other nine stations and the
# post-2024 range get used, pivoted into wide columns.
feat_v2 = (bronze
    .select("station", "date", "AWND", "TMAX", "TMIN", "PRCP")
    .withColumn("PRCP_today", col("PRCP"))                       # today's rain (now a legit feature for t+1)
    .withColumn("PRCP_lag1",  lag("PRCP", 1).over(w))            # yesterday's rain
    .withColumn("PRCP_roll3", avg("PRCP").over(w.rowsBetween(-2, 0)))  # 3-day avg
    .withColumn("TMAX_lag1",  lag("TMAX", 1).over(w))
    .na.drop(subset=["station", "date"]))

FT2 = f"{CATALOG}.features.weather_daily_v2"
spark.sql(f"DROP TABLE IF EXISTS {FT2}")
fe.create_table(name=FT2, primary_keys=["date", "station"], timestamp_keys=["date"],
                df=feat_v2, description="v2 — adds lagged/rolling PRCP + temp")

print(f"v2 built: {feat_v2.count()} rows")